# Ninai × Google ADK Adapter

Integrates the **Ninai Python SDK** with **Google Agent Development Kit (ADK)**:

1. Ninai tools wrapped as `FunctionTool` for ADK agents
2. `NinaiADKMemoryService` — persists ADK session events in Ninai
3. End-to-end tool invocation smoke test

All API calls are **mocked** — runs offline without a live Ninai server.

In [1]:
import sys, os, warnings
warnings.filterwarnings('ignore')
SDK_PATH = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'sdk', 'python'))
if SDK_PATH not in sys.path:
    sys.path.insert(0, SDK_PATH)
import google.adk, ninai
print('google-adk:', google.adk.__version__)
print('ninai SDK:', ninai.__version__)


google-adk: 1.29.0
ninai SDK: 1.0.0


## Mock Ninai client

In [2]:
from unittest.mock import MagicMock
from types import SimpleNamespace

_STORE: dict = {}
_CTR = [0]

def _mock_create(**kwargs):
    _CTR[0] += 1
    m = SimpleNamespace(id=str(_CTR[0]), content=kwargs.get('content', ''),
                        title=kwargs.get('title', ''), tags=kwargs.get('tags', []))
    _STORE[m.id] = m
    return m

def _mock_search(query, **kwargs):
    items = [SimpleNamespace(memory_id=m.id, content=m.content, score=0.9, title=m.title)
             for m in list(_STORE.values())[-5:]]
    return SimpleNamespace(items=items[:3], total=len(items))

ninai_client = MagicMock()
ninai_client.memories.create.side_effect = _mock_create
ninai_client.memories.search.side_effect = _mock_search
print('Mock Ninai client ready')


Mock Ninai client ready


## 1. Wrap Ninai as ADK FunctionTools

In [3]:
from google.adk.tools import FunctionTool


def ninai_search_memory(query: str, top_k: int = 5) -> dict:
    """Search Ninai organisational memory and return relevant context.

    Args:
        query: Natural-language search query.
        top_k: Maximum number of results.

    Returns:
        dict with 'results' (list of content strings) and 'count'.
    """
    res = ninai_client.memories.search(query, limit=top_k)
    return {
        'results': [r.content for r in res.items],
        'count': len(res.items),
    }


def ninai_store_memory(content: str, tags: str = '') -> dict:
    """Store a fact or decision in Ninai organisational memory.

    Args:
        content: The text to remember.
        tags: Comma-separated tag list.

    Returns:
        dict with 'memory_id' and 'status'.
    """
    tag_list = [t.strip() for t in tags.split(',') if t.strip()]
    mem = ninai_client.memories.create(content=content, tags=tag_list)
    return {'memory_id': mem.id, 'status': 'stored'}


# Wrap as ADK FunctionTool
search_tool = FunctionTool(ninai_search_memory)
remember_tool = FunctionTool(ninai_store_memory)

print(f'ADK tools created: {search_tool.name}, {remember_tool.name}')


ADK tools created: ninai_search_memory, ninai_store_memory


## 2. Build an ADK Agent with Ninai tools

In [4]:
from google.adk import Agent
from google.adk.sessions import InMemorySessionService

# Build an ADK agent with Ninai tools
agent = Agent(
    name='ninai_enterprise_agent',
    model='gemini-2.0-flash',          # replace with your deployed model
    description='Enterprise assistant with Ninai memory',
    instruction=(
        'You are an enterprise assistant. '
        'Use ninai_store_memory to remember important facts. '
        'Use ninai_search_memory to retrieve context before answering.'
    ),
    tools=[search_tool, remember_tool],
)

print(f'Agent: {agent.name}')
print(f'Tools: {[t.name for t in agent.tools]}')


Agent: ninai_enterprise_agent
Tools: ['ninai_search_memory', 'ninai_store_memory']


## 3. Tool invocation smoke test

In [5]:
# Demonstrate tool invocations directly (no live LLM call needed)
store_result = ninai_store_memory('Board approved $10M Series A on 2026-04-01', tags='finance,funding')
print('Stored:', store_result)

store_result2 = ninai_store_memory('Engineering headcount target: 25 by Q4', tags='hr,headcount')
print('Stored:', store_result2)

search_result = ninai_search_memory('funding round')
print('Search hits:', search_result['count'])
for r in search_result['results']:
    print(f'  - {r}')

print('\nADK + Ninai adapter verified.')


Stored: {'memory_id': '1', 'status': 'stored'}
Stored: {'memory_id': '2', 'status': 'stored'}
Search hits: 2
  - Board approved $10M Series A on 2026-04-01
  - Engineering headcount target: 25 by Q4

ADK + Ninai adapter verified.


## 4. NinaiADKMemoryService

In [6]:
# Optional: NinaiADKMemoryService — stores ADK session events in Ninai
# Useful for cross-session recall across agents.

class NinaiADKMemoryService:
    """
    Thin adapter: persist ADK session events into Ninai memory so that
    a future ADK agent can search prior sessions via ninai_search_memory.
    """

    def __init__(self, ninai_client):
        self._client = ninai_client

    def save_session_event(self, session_id: str, event: dict) -> None:
        content = event.get('content') or str(event)
        self._client.memories.create(
            content=content,
            title=f'ADK event [{event.get("role", "?")}, session:{session_id}]',
            tags=['adk', f'session:{session_id}', event.get('role', 'unknown')],
        )

    def search(self, query: str, limit: int = 5) -> list:
        res = self._client.memories.search(query, limit=limit)
        return [r.content for r in res.items]


svc = NinaiADKMemoryService(ninai_client=ninai_client)
svc.save_session_event('sess-99', {'role': 'user', 'content': 'Schedule Q3 planning call'})
svc.save_session_event('sess-99', {'role': 'agent', 'content': 'Q3 planning call scheduled for 2026-06-01'})

hits = svc.search('Q3 planning')
print(f'Recalled {len(hits)} events for "Q3 planning":')
for h in hits:
    print(f'  {h}')


Recalled 3 events for "Q3 planning":
  Board approved $10M Series A on 2026-04-01
  Engineering headcount target: 25 by Q4
  Schedule Q3 planning call
